In [1]:


from src.utils.file_utils import parse_file_name
from migration.cur.decomposer import CurSqlDecomposer, CurDecomposerWriter
from migration.cur.metadata import CurMetadataProcessor
from migration.cur.generator import CurPySparkGenerator
from src.paths import *

In [2]:
USERNAME

'dungp'

In [3]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_r_mhbos_m_client_crs.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_mhbos_m_client.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()["cur"]

print(f"File gốc tại: {input_file}")

File gốc tại: C:\Users\dungp\projects\datalake-script\dml\cur\cur_dim_contact.sql


In [4]:
# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = CurSqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
writer = CurDecomposerWriter()
writer.write(decomposed_script, output_root / file_name)
print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_contact\processing_steps


In [5]:
"mhbos"

'mhbos'

In [6]:
decomposed_script.source_blocks["MHBOS"].interacted_temp_tables

{'TEMP_DIM_ACCOUNT_CONTACT',
 'TEMP_DIM_BRANCH_CONTACT',
 'TEMP_DIM_TRADER_CONTACT'}

In [7]:

# ==========================================
# BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
# ==========================================
processor = CurMetadataProcessor(source_rules)

try:
    # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
    pipeline_config = processor.process(decomposed_script, input_file)

    # Ghi file YAML
    metadata_output_dir = output_root / file_name / "metadata"
    processor.write_yaml(pipeline_config, metadata_output_dir)

    print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
    print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
    print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['primary_key']['logical_primary_key']}")
    print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

except FileNotFoundError as e:
    print(f"❌ [Lỗi Bước 2]: {e}")
    print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model UNKNOWN
   -> Khóa (Key) nhận diện được: ['OWNER_ID', 'CONTACT_OWNER_TYPE', 'CONTACT_TYPE']
   -> File YAML đã lưu tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_contact\metadata\cur_dim_contact.yaml


In [8]:
print("==========================================")
print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
print("==========================================")

with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
    pipeline_config = yaml.safe_load(f)

generator = CurPySparkGenerator(source_rules, output_mode="simple")
ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)

print("🎉 Hoàn tất toàn bộ Pipeline!")

 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
Reading pre-processing SQL from: 88_unknown_blocks
Generated DML from 'model_1/cur_dml.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_contact\model_1\dml\cur_dim_contact_unknown_source_cur_dml.py
Generated DML from 'model_2a/cur_dml.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_contact\model_2a\dml\cur_dim_contact_unknown_source_cur_dml.py
Generated DML from 'model_2b/cur_dml.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_contact\model_2b\dml\cur_dim_contact_unknown_source_cur_dml.py
Generated DML from 'model_4/cur_dml_nopartition.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_contact\model_4\dml\cur_dim_contact_unknown_source_cur_dml_nopartition.py
Generated DML from 'model_4/cur_dml_withpartition.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\cur_dim_contact\model_4\dml\cur_dim_contact_unknown_source_cur_dml_w

In [9]:
pipeline_config

{'pipeline_id': 'cur_dim_contact',
 'layer': 'cur',
 'model_type': 'UNKNOWN',
 'target_table_name': 'DIM_CONTACT',
 'columns': [{'name': 'owner_id', 'type': 'VARCHAR(50)', 'remark': None},
  {'name': 'contact_owner_type', 'type': 'VARCHAR(20)', 'remark': None},
  {'name': 'contact_type', 'type': 'VARCHAR(10)', 'remark': None},
  {'name': 'contact_value', 'type': 'VARCHAR(150)', 'remark': None},
  {'name': 'contact_name', 'type': 'VARCHAR(100)', 'remark': None},
  {'name': 'contact_create_date', 'type': 'DATE', 'remark': None},
  {'name': 'contact_update_date', 'type': 'DATE', 'remark': None},
  {'name': 'line_of_business', 'type': 'VARCHAR(20)', 'remark': None},
  {'name': 'source_name',
   'type': 'VARCHAR(10)',
   'remark': 'non_original_field'},
  {'name': 'source_record_id', 'type': 'VARCHAR(50)', 'remark': None},
  {'name': 'sequence_no', 'type': 'INT', 'remark': None},
  {'name': 'etl_timestamp', 'type': 'STRING', 'remark': 'non_original_field'}],
 'primary_key': {'logical_primar

In [10]:
dml_context["main_processing_sqls"]

KeyError: 'main_processing_sqls'

In [ ]:

# Read pre_processing SQLs
pre_processing_sqls = []
for step in pipeline_config.get("pre_processing", []):
    if step.get("action") == "skip":
        continue
    step_file = Path(step["file"])
    if step_file.exists():
        pre_processing_sqls.append(step_file.read_text(encoding="utf-8"))